In [ ]:
import logging

from numpy import isclose
from pathlib import Path
from rdflib import URIRef, Literal

logging.basicConfig(level=logging.DEBUG, force=True)
logger = logging.getLogger()

path_data = Path("notebooks/data")

## Load (graph) data

In [ ]:
simulated_scenario_log_file = "example_1_1_event_log.json"
event_data_file = path_data.joinpath(simulated_scenario_log_file)

event_data_file = Path(
    r"directory\aggregated_event_data\logs\example_material_2_event_log.json"
)

from aggregated_traces.utils.construct_ekg import load_rdf_graph

ekg_graph_rdf = load_rdf_graph(event_data_file)

In [ ]:
from aggregated_traces.utils.construct_ekg import insert_DF_DP

ekg_graph_rdf = insert_DF_DP(ekg_graph_rdf)

## Insert fractions
**Note that this requires a complete trace graph!**<br>
Including all incoming and/or outgoing quantities for each lot in the path.

In [ ]:
# Check incoming vs outgoing amount
from aggregated_traces.utils.construct_ekg import check_quantities

query_result = check_quantities(ekg_graph_rdf)
print(query_result.serialize(format="txt").decode())

In [ ]:
# Insert fractions
from aggregated_traces.utils.construct_ekg import insert_fractions

ekg_graph_rdf = insert_fractions(ekg_graph_rdf)

## Create NetworkX graph

In [ ]:
from aggregated_traces.utils.construct_ekg import generate_networkx_di_graph

ekg_graph_nx = generate_networkx_di_graph(ekg_graph_rdf)

## Visualization

In [ ]:
from aggregated_traces.utils.visualization import generate_graph_visualization

fig = generate_graph_visualization(ekg_graph_nx, f"output/{event_data_file.stem}")

# Traceability questions

In [ ]:
from pandas import set_option

set_option("display.max_colwidth", None)

from aggregated_traces.utils.ekg_analysis import compute_trace_probabilities

### Backward
* What **equipment** was used to produce devices in packing unit X?
* With what propability was **equipment** used to produce an arbitrary device in packing unit X?

In [ ]:
# Define recources for which to retrieve the trace
entities_source_backward = [
    URIRef("http://example.org/id/ekg/aggregated_traces/Lot2_0_Pack0"),
]

In [ ]:
df_backward, edges_paths_backward = compute_trace_probabilities(
    rdf_trace_graph=ekg_graph_rdf,
    nx_trace_graph=ekg_graph_nx,
    source_entities=entities_source_backward,
    trace_backward=True,
)

if not all(
    isclose(
        df_backward.groupby(["entity_source", "product_model"])["probability"]
        .sum()
        .astype(float),
        1,
    )
):
    raise Warning("Sum of probabilities by production step do not add up to one!")

df_backward.groupby(["entity_source", "entity_target"])[["probability"]].sum()

### Forward

* What packing units might be impacted by an issue on **equipment** (in a given time window)?
* With what probability did a device produced on **equipment** (in a given time window) end up in packing unit X?*

In [ ]:
# Find input lots
results = ekg_graph_rdf.query("""
PREFIX : <http://example.org/def/ekg/aggregated_traces/>

SELECT DISTINCT ?MaterialLot
WHERE {
    [] :bizStep "assembling" ;
        :inputQuantity/:class ?MaterialLot .

    ?MaterialLot a :MaterialLot .
}
""")

print(results.serialize(format="txt").decode())

In [ ]:
# Find input lots used for multiple lots
results = ekg_graph_rdf.query("""
PREFIX : <http://example.org/def/ekg/aggregated_traces/>

SELECT DISTINCT ?MaterialLot
WHERE {
    ?event :bizStep "assembling" ;
        :inputQuantity/:class ?MaterialLot .

    ?MaterialLot a :MaterialLot .
}
GROUP BY ?MaterialLot
HAVING (count(distinct ?event) > 1)
""")

print(results.serialize(format="txt").decode())

In [ ]:
from rdflib import Variable

entities_source_forward = [b[Variable("MaterialLot")] for b in results.bindings]

entities_source_forward_window = []

In [ ]:
# Find locations and time window
print(
    ekg_graph_rdf.query("""
PREFIX : <http://example.org/def/ekg/aggregated_traces/>

SELECT DISTINCT
    ?location
    (group_concat(round(?timestamp*100)/100) as ?timestamps)
WHERE {
    [] :bizStep "departing" ;
        :timestamp ?timestamp ;
        :location ?location .
}
GROUP BY ?location
""")
    .serialize(format="txt")
    .decode()
)

In [ ]:
entities_source_forward_window = [
    (
        URIRef("http://example.org/id/ekg/aggregated_traces/DB1"),
        (Literal(7), Literal(10)),
    )
]

entities_source_forward = []

In [ ]:
entities_source_forward = [
    URIRef("http://example.org/id/ekg/aggregated_traces/WBM1_0"),
]

In [ ]:
if entities_source_forward_window:
    df_forward, edges_paths_forward = compute_trace_probabilities(
        rdf_trace_graph=ekg_graph_rdf,
        nx_trace_graph=ekg_graph_nx,
        source_entities_time=entities_source_forward_window,
        trace_backward=False,
    )
else:
    df_forward, edges_paths_forward = compute_trace_probabilities(
        rdf_trace_graph=ekg_graph_rdf,
        nx_trace_graph=ekg_graph_nx,
        source_entities=entities_source_forward,
        trace_backward=False,
    )


if not all(
    isclose(
        df_backward.groupby(["entity_source", "node_source"])["probability"]
        .sum()
        .astype(float),
        1,
    )
):
    raise Warning("Sum of probabilities by source event do not add up to one!")

df_forward.groupby(["entity_source", "entity_target"])[["probability"]].sum()

In [ ]:
df_forward

### Visualization paths

In [ ]:
fig_paths = generate_graph_visualization(
    ekg_graph_nx,
    f"output/{event_data_file.stem}",
    edges_backward=edges_paths_backward,
    edges_forward=edges_paths_forward,
)

# Statistics

In [ ]:
from scipy.stats import entropy

## Uniformity of probabilities

In [ ]:
backward = True

if backward:
    df_trace = df_backward
    group_key = "product_model"  # "entity_source"
else:
    df_trace = df_forward
    group_key = "entity_source"  # "node_source"


for group_value in df_trace[group_key].unique():
    df_group = df_trace[df_trace[group_key] == group_value]

    n_targets = df_group["entity_target"].nunique()
    probabilities = df_group.groupby(["entity_source", "entity_target"])[
        "probability"
    ].sum()
    probabilities = probabilities.astype(float)

    kl_divergence = entropy(pk=probabilities.values, qk=[1 / n_targets] * n_targets)
    print(
        f"{group_key}={group_value}: Kullback-Leibler divergence={kl_divergence}"
    )  # closer to 0 is closer to uniform distribution